# Linear regression: simple to multiple

Linear regression fits a straight relationship between predictors and a
continuous target. We start with **one** predictor (a picture you can draw),
then add more and meet the issues that come with them: multicollinearity,
R²'s blind spot, and how to compare coefficients. We use `smartcore`'s
`LinearRegression` (it fits simple and multiple regression identically — the
*interpretation* is what changes).

This builds on the `ndarray` and `polars` [foundations](../01-foundations/ndarray-basics.ipynb).

In [ ]:
:dep smartcore = { version = "0.3" }
:dep plotters = { version = "0.3", default-features = false, features = ["evcxr", "all_series", "all_elements"] }
:dep plotters-statistical = { version = "0.2.0" }
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::linalg::basic::arrays::Array;

// 60 samples. True relationship: y = 2*x1 + 3*x2 + small noise.
// x3 is ~collinear with x1 (used for multicollinearity); xn is pure noise.
let n = 60usize;
let x1: Vec<f64> = (0..n).map(|i| (i as f64 * 0.17).sin() * 3.0 + i as f64 * 0.15).collect();
let x2: Vec<f64> = (0..n).map(|i| (i as f64 * 0.23).cos() * 2.0 + 5.0).collect();
let x3: Vec<f64> = (0..n).map(|i| x1[i] + (((i * 7) % 5) as f64) * 0.1 - 0.2).collect();
let xn: Vec<f64> = (0..n).map(|i| (((i * 29) % 13) as f64) - 6.0).collect();
let y:  Vec<f64> = (0..n).map(|i| 2.0 * x1[i] + 3.0 * x2[i] + (((i * 11) % 7) as f64 - 3.0) * 0.2).collect();

// Build a design matrix from chosen feature columns (smartcore adds the intercept).
fn design(cols: &[&[f64]]) -> DenseMatrix<f64> {
    let (n, p) = (cols[0].len(), cols.len());
    let mut flat = Vec::with_capacity(n * p);
    for i in 0..n { for c in cols { flat.push(c[i]); } }
    DenseMatrix::new(n, p, flat, false)
}
println!("{} samples; features x1, x2, x3(~x1), xn(noise)", n);

## Simple regression: one predictor

Start with the single strongest predictor of `y` (in a real project you'd read
this off the [EDA correlation heatmap](../01b-eda/exploratory-data-analysis.ipynb) —
here it's `x1`). With one predictor the model is a line `y = β·x1 + intercept`,
and the coefficient means *"y changes by β for each unit of x1."*

In [ ]:
{
    use smartcore::linear::linear_regression::LinearRegression;
    use smartcore::metrics::r2;
    let x = design(&[&x1]);
    let lr = LinearRegression::fit(&x, &y, Default::default()).unwrap();
    println!("simple: y = {:.2}*x1 + {:.2}", lr.coefficients().get((0, 0)), lr.intercept());
    println!("R^2 = {:.3}  (one predictor can't explain all of y, since y also depends on x2)", r2(&y, &lr.predict(&x).unwrap()));
}

## Multiple regression: many predictors

Add `x2`. The fit is now a **hyperplane**, not a line — there's no single 2-D
picture to draw, so we read the coefficients instead. Crucially, each coefficient
now means *"the effect of this predictor **holding all others constant**"* — it
has already netted out the other predictors' influence, which a simple regression
ignores entirely.

In [ ]:
{
    use smartcore::linear::linear_regression::LinearRegression;
    use smartcore::metrics::r2;
    let x = design(&[&x1, &x2]);
    let lr = LinearRegression::fit(&x, &y, Default::default()).unwrap();
    println!("multiple: y = {:.2}*x1 + {:.2}*x2 + {:.2}", lr.coefficients().get((0, 0)), lr.coefficients().get((1, 0)), lr.intercept());
    println!("R^2 = {:.3}  (recovers the true 2*x1 + 3*x2)", r2(&y, &lr.predict(&x).unwrap()));
}

### Residual diagnostics

Coefficients and R² don't tell you *whether the linear form fits*. The
**residuals** (actual − fitted) should scatter with no pattern around zero — a
curve or funnel would betray a missing non-linear term or non-constant variance.
`plotters-statistical`'s `ResidualPlot::from_residuals(fitted, residuals)` draws
them against the fitted values with a zero line and a moving-average **trend**. On
this well-specified `y = 2·x1 + 3·x2` data the trend stays flat — what a correctly
specified model should look like:

In [ ]:
{
    use smartcore::linear::linear_regression::LinearRegression;
    use plotters::prelude::*;
    use plotters_statistical::ResidualPlot;
    let x = design(&[&x1, &x2]);
    let lr = LinearRegression::fit(&x, &y, Default::default()).unwrap();
    let fitted: Vec<f64> = lr.predict(&x).unwrap();
    let residuals: Vec<f64> = y.iter().zip(fitted.iter()).map(|(t, f)| t - f).collect();
    let (flo, fhi) = (fitted.iter().cloned().fold(f64::INFINITY, f64::min),
                      fitted.iter().cloned().fold(f64::NEG_INFINITY, f64::max));
    let rlim = residuals.iter().cloned().fold(0.0, |m, v| f64::max(m, v.abs())) * 1.2 + 1e-9;
    evcxr_figure((520, 320), |root| {
        root.fill(&WHITE)?;
        let mut c = ChartBuilder::on(&root)
            .caption("residuals vs fitted", ("sans-serif", 15))
            .margin(10).x_label_area_size(32).y_label_area_size(44)
            .build_cartesian_2d(flo..fhi, (-rlim)..rlim)?;
        c.configure_mesh().x_desc("fitted value").y_desc("residual").draw()?;
        c.draw_series(std::iter::once(ResidualPlot::from_residuals(&fitted, &residuals)?.trend(true)))?;
        Ok(())
    })
}

## Multicollinearity & VIF

`x3` was built to be nearly identical to `x1` (correlated *with another
predictor*, not just with the target). Including both makes their individual
coefficients **unstable** — the model can't tell which one deserves the credit —
even if overall predictions stay fine.

The **Variance Inflation Factor** quantifies this: `VIF_j = 1 / (1 − R²_j)`, where
`R²_j` comes from regressing predictor *j* on all the *other* predictors. It has
no maintained crate, so we compute it by hand — reusing this very chapter's
regression tooling to diagnose itself (VIF > 5–10 signals a problem):

In [ ]:
{
    use smartcore::linear::linear_regression::LinearRegression;
    use smartcore::metrics::r2;
    let feats: Vec<(&str, Vec<f64>)> = vec![("x1", x1.clone()), ("x2", x2.clone()), ("x3", x3.clone())];
    for j in 0..feats.len() {
        let target = &feats[j].1;
        let others: Vec<&[f64]> = (0..feats.len()).filter(|&k| k != j).map(|k| feats[k].1.as_slice()).collect();
        let x = design(&others);
        let lr = LinearRegression::fit(&x, target, Default::default()).unwrap();
        let r2j = r2(target, &lr.predict(&x).unwrap());
        println!("VIF({}) = {:.1}", feats[j].0, 1.0 / (1.0 - r2j));
    }
    println!("-> x1 and x3 have huge VIFs (they're nearly the same feature); x2 is fine.");
}

The fix depends on your goal: **drop** one of the collinear pair, **combine**
them, or **accept** it if you only care about predictions (not about reading the
coefficients). Those are genuinely different goals with different right answers.

## Adjusted R² vs. plain R²

Plain R² can only go **up** as you add predictors — even useless ones. Add the
pure-noise column `xn` and watch:

In [ ]:
{
    use smartcore::linear::linear_regression::LinearRegression;
    use smartcore::metrics::r2;
    // adjusted R^2 = 1 - (1-R^2)*(n-1)/(n-p-1)
    let adj = |r2v: f64, p: usize| 1.0 - (1.0 - r2v) * (n as f64 - 1.0) / (n as f64 - p as f64 - 1.0);

    let x_a = design(&[&x1, &x2]);
    let r2_a = r2(&y, &LinearRegression::fit(&x_a, &y, Default::default()).unwrap().predict(&x_a).unwrap());
    let x_b = design(&[&x1, &x2, &xn]);
    let r2_b = r2(&y, &LinearRegression::fit(&x_b, &y, Default::default()).unwrap().predict(&x_b).unwrap());

    println!("without noise col: R^2 = {:.5}, adjusted R^2 = {:.5}", r2_a, adj(r2_a, 2));
    println!("with    noise col: R^2 = {:.5}, adjusted R^2 = {:.5}", r2_b, adj(r2_b, 3));
    println!("-> plain R^2 ticked UP for the noise column; adjusted R^2 did not reward it.");
}

## Standardized coefficients — the linear model's "feature importance"

Raw coefficient magnitudes aren't comparable when predictors are on different
scales. Fit on **z-scored** predictors (the standardization from the
[ETL chapter](../01c-etl/data-preparation.ipynb)) and the coefficients become
directly comparable — a horizontal bar chart of their magnitudes is the linear
model's answer to the tree chapters' feature-importance plots:

In [ ]:
{
    use smartcore::linear::linear_regression::LinearRegression;
    use plotters::prelude::*;
    let zscore = |v: &[f64]| { let m = v.iter().sum::<f64>() / v.len() as f64; let sd = (v.iter().map(|x| (x - m).powi(2)).sum::<f64>() / v.len() as f64).sqrt(); v.iter().map(|x| (x - m) / sd).collect::<Vec<f64>>() };
    let (z1, z2, zn) = (zscore(&x1), zscore(&x2), zscore(&xn));
    let x = design(&[&z1, &z2, &zn]);
    let lr = LinearRegression::fit(&x, &y, Default::default()).unwrap();
    let coefs = [("x1", lr.coefficients().get((0,0)).abs()), ("x2", lr.coefficients().get((1,0)).abs()), ("xn", lr.coefficients().get((2,0)).abs())];
    for (nm, c) in &coefs { println!("|standardized coef| {} = {:.3}", nm, c); }
    let maxc = coefs.iter().map(|c| c.1).fold(0.0, f64::max) * 1.1;
    evcxr_figure((420, 200), |root| {
        root.fill(&WHITE)?;
        let mut chart = ChartBuilder::on(&root).caption("Standardized coefficients", ("sans-serif", 16)).margin(8).x_label_area_size(30).y_label_area_size(30).build_cartesian_2d(0f64..maxc, 0i32..3i32)?;
        chart.configure_mesh().draw()?;
        chart.draw_series((0..3).map(|i| Rectangle::new([(0.0, i as i32), (coefs[i].1, i as i32 + 1)], BLUE.filled())))?;
        Ok(())
    })
}

`x2` has the largest standardized coefficient (its true weight is 3 vs. x1's 2),
and the noise column `xn` is near zero — exactly right.

## Interaction & polynomial terms (a taste)

Linear regression can capture non-linear effects if you *engineer* the features:
add a product `x1·x2` (interaction) or a square `x1²` (polynomial), then refit.
A fuller treatment belongs in the [ETL / feature-engineering
chapter](../01c-etl/data-preparation.ipynb); here's just the idea:

In [ ]:
{
    use smartcore::linear::linear_regression::LinearRegression;
    use smartcore::metrics::r2;
    let adj = |r2v: f64, p: usize| 1.0 - (1.0 - r2v) * (n as f64 - 1.0) / (n as f64 - p as f64 - 1.0);
    let inter: Vec<f64> = (0..n).map(|i| x1[i] * x2[i]).collect();
    let sq: Vec<f64> = (0..n).map(|i| x1[i] * x1[i]).collect();

    let fit_r2 = |cols: &[&[f64]]| { let x = design(cols); r2(&y, &LinearRegression::fit(&x, &y, Default::default()).unwrap().predict(&x).unwrap()) };
    let (r_simple, p_s) = (fit_r2(&[&x1]), 1);
    let (r_multi, p_m)  = (fit_r2(&[&x1, &x2]), 2);
    let (r_inter, p_i)  = (fit_r2(&[&x1, &x2, &inter, &sq]), 4);
    println!("{:<28} {:>7} {:>12}", "model", "R^2", "adjusted R^2");
    println!("{:<28} {:>7.3} {:>12.3}", "simple (x1)", r_simple, adj(r_simple, p_s));
    println!("{:<28} {:>7.3} {:>12.3}", "multiple (x1,x2)", r_multi, adj(r_multi, p_m));
    println!("{:<28} {:>7.3} {:>12.3}", "+ interaction + x1^2", r_inter, adj(r_inter, p_i));
}

The comparison table is the payoff: adding `x2` is a real gain; the interaction/
polynomial terms barely move adjusted R² here (the true relationship was linear).
For a properly cross-validated version of this comparison, see the
[metrics deep-dive](../01d-evaluation/metrics-deep-dive.ipynb).

Next: [multi-output regression](multi-output-regression.ipynb) — many predictors
to *many* targets at once, a different problem shape entirely.